# Nurse Navigation - Repeat Callers

Measures how often the same person generates another 911 call transferred to nurse navigation within 30, 60, and 90 days.

Logis has no single patient identifier, so each person is resolved from patient first name, last name, date of birth, and phone. Exact matches are applied first, followed by tightly limited fuzzy rules for spelling differences, nicknames, and last-name changes. Every link is tagged with the rule that produced it, so link counts can be reviewed by rule.

Definitions:

- Index call: any call from an identified person.
- Repeat (return): a separate 911 incident from the same person within the window after the index call.
- Return rate: share of index calls followed by a return within the window. Only index calls with a full follow-up window before the end of the data are counted, so calls in the final 30, 60, or 90 days do not lower the rate.

No names, dates of birth, or phone numbers are written to the Excel output. Persons are labeled with a generated person key.

## 1. Setup

In [ ]:
import os, re, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200); pd.set_option("display.max_colwidth", 200)
plt.rcParams.update({"figure.figsize":(11,5),"figure.dpi":110,"axes.grid":True,"grid.alpha":0.25,
    "axes.spines.top":False,"axes.spines.right":False,"font.size":11,"axes.titlesize":13,"axes.titleweight":"bold"})
TEAL, NAVY, CORAL, GOLD, GREY = "#028090","#0B2545","#D1495B","#E0A500","#8FA0A6"

DATA_DIR   = "/Workspace/Users/josh.smitherman@gmr.net/nurse_nav/data"
OUT_DIR    = "/Workspace/Users/josh.smitherman@gmr.net/nurse_nav/results"
PROMPT_DIR = "/Workspace/Users/josh.smitherman@gmr.net/nurse_nav/prompts"
SOURCE_FILE = "data_april2026-aug2026.xlsx"

WINDOWS = [30, 60, 90]
FREQUENT_THRESHOLDS = [2, 3, 5]
NAME_MIN_SIMILARITY = 0.88
MAX_LAST_NAMES_PER_DOB = 25
MAX_PEOPLE_PER_PHONE = 5
USE_PERSONAL_ID = False
QA_SAMPLE_N = 60
RANDOM_SEED = 42

os.makedirs(OUT_DIR, exist_ok=True)
RUN_ID = pd.Timestamp.now().strftime("%Y%m%d_%H%M")
RESULTS = {}
def keep(d, n): RESULTS[n] = d.copy(); return d
print("run:", RUN_ID)

## 2. Load calls and resolve columns

In [ ]:
def clean_col(c): return re.sub(r"_+","_",re.sub(r"[^\w]+","_",str(c).strip())).lower()
raw = pd.read_excel(os.path.join(DATA_DIR, SOURCE_FILE)); raw.columns = [clean_col(c) for c in raw.columns]
def find_col(df, exact, contains=None):
    norm = lambda x: x.strip("_"); nrm = {norm(c): c for c in df.columns}
    for c in exact:
        if c in df.columns: return c
        if norm(c) in nrm: return nrm[norm(c)]
    for pat in (contains or []):
        hits = [c for c in df.columns if pat in c]
        if hits: return sorted(hits, key=len)[0]
    return None

DATE        = find_col(raw, ["transaction_create_date_time_eastern"], ["date_time_eastern"])
FNAME       = find_col(raw, ["patientfname","patient_first_name"], ["fname"])
LNAME       = find_col(raw, ["patientlname","patient_last_name"], ["lname"])
DOB         = find_col(raw, ["dateofbirth","date_of_birth","dob"], ["birth"])
PHONE       = find_col(raw, ["phone","patient_phone"])
INCIDENT    = find_col(raw, ["911_id"], ["911_id"])
REC_ID      = find_col(raw, ["id"])
PID         = find_col(raw, ["personal_id_number"], ["personal_id"])
MARKET      = find_col(raw, ["market_name","market"], ["market"])
DISPO       = find_col(raw, ["transaction_response_names"], ["response_name"])
CALLER_TYPE = find_col(raw, ["caller_type"], ["caller_type"])
NOTES       = find_col(raw, ["nurses_notes","nurse_notes","notes"], ["note"])

df = raw.copy()
df["call_dt"] = pd.to_datetime(df[DATE], errors="coerce")
no_date = int(df["call_dt"].isna().sum())
df = df[df["call_dt"].notna()].sort_values("call_dt").reset_index(drop=True)
df["record_id"] = df[REC_ID].astype(str) if REC_ID else df.index.astype(str)
print(f"{len(raw):,} calls loaded; {no_date:,} without a valid call date removed; {len(df):,} remain")
print(f"call dates: {df['call_dt'].min():%Y-%m-%d} to {df['call_dt'].max():%Y-%m-%d}")
fields = pd.DataFrame({"field":["call date","patient first name","patient last name","date of birth","patient phone",
                                "911 incident","record","personal identification number","market","disposition",
                                "caller type","nurse notes"],
                       "resolved":[DATE,FNAME,LNAME,DOB,PHONE,INCIDENT,REC_ID,PID,MARKET,DISPO,CALLER_TYPE,NOTES]})
keep(fields, "field_mapping")
fields

## 3. Identity field profile

Fill rate and distinct values for each field used to identify a person. A high count on the most common value indicates a placeholder entry (for example, a default date of birth or a facility phone).

In [ ]:
BLANKS = {"", "nan", "none", "null", "na", "n/a", "unknown", "unk", "nat"}
def is_blank(s): return s.isna() | s.astype(str).str.strip().str.lower().isin(BLANKS)

prof = []
for label, col in [("patient first name",FNAME),("patient last name",LNAME),("date of birth",DOB),
                   ("patient phone",PHONE),("911 incident",INCIDENT),("personal identification number",PID)]:
    if col:
        b = is_blank(df[col])
        vc = df.loc[~b, col].astype(str).str.strip().str.lower().value_counts()
        prof.append({"field":label, "column":col, "filled_percent":round((~b).mean()*100,1),
                     "distinct_values":int(vc.size), "calls_on_most_common_value":int(vc.iloc[0]) if len(vc) else 0})
profile = pd.DataFrame(prof)
keep(profile, "identity_profile")
profile

## 4. Remove duplicate rows for the same 911 incident

Rows that share a 911 incident number describe the same incident. The earliest row is kept so a single incident is not counted as a repeat of itself.

In [ ]:
before = len(df)
if INCIDENT:
    has_inc = ~is_blank(df[INCIDENT])
    dup = has_inc & df[INCIDENT].astype(str).str.strip().duplicated(keep="first")
    df = df[~dup].reset_index(drop=True)
removed_dup = before - len(df)
print(f"duplicate incident rows removed: {removed_dup:,}; calls remaining: {len(df):,}")

## 5. Standardize names, date of birth, and phone

Names are lowercased with punctuation and suffixes (jr, sr, ii, iii, iv) removed. Placeholder names, placeholder dates of birth, and phones shared by many different people (for example, a facility line) are excluded from linking.

In [ ]:
SUFFIXES = {"jr","sr","ii","iii","iv"}
BAD_FIRST = {"unknown","unk","patient","pt","na","none","test","nan","refused","anonymous","caller","baby","infant","male","female"}
BAD_LAST  = {"doe","unknown","unk","patient","na","none","test","nan","refused","anonymous","caller"}
NICKNAMES = {"bob":"robert","bobby":"robert","rob":"robert","robby":"robert","bill":"william","billy":"william","will":"william",
    "liz":"elizabeth","beth":"elizabeth","betty":"elizabeth","jim":"james","jimmy":"james","mike":"michael","dave":"david",
    "tom":"thomas","tommy":"thomas","tony":"anthony","joe":"joseph","joey":"joseph","kate":"katherine","kathy":"katherine",
    "cathy":"catherine","chris":"christopher","dan":"daniel","danny":"daniel","steve":"steven","rick":"richard","ricky":"richard",
    "dick":"richard","sue":"susan","peggy":"margaret","maggie":"margaret","jen":"jennifer","jenny":"jennifer","larry":"lawrence",
    "patty":"patricia","ed":"edward","eddie":"edward","ted":"edward","charlie":"charles","chuck":"charles","sam":"samuel",
    "alex":"alexander","nick":"nicholas","matt":"matthew","andy":"andrew","greg":"gregory","ron":"ronald","don":"donald",
    "ken":"kenneth","jerry":"gerald","debbie":"deborah","deb":"deborah","vicky":"victoria","becky":"rebecca","terry":"terrence"}

def name_tokens(x):
    if pd.isna(x): return []
    return [t for t in re.sub(r"[^a-z ]", " ", str(x).lower()).split() if t not in SUFFIXES]

df["fn"] = df[FNAME].apply(lambda x: (name_tokens(x) or [""])[0]) if FNAME else ""
df["ln"] = df[LNAME].apply(lambda x: "".join(name_tokens(x))) if LNAME else ""
df["fn_canon"] = df["fn"].map(lambda f: NICKNAMES.get(f, f))
df["name_ok"] = (df["fn"].str.len() >= 2) & (df["ln"].str.len() >= 2) & ~df["fn"].isin(BAD_FIRST) & ~df["ln"].isin(BAD_LAST)

dob = pd.to_datetime(df[DOB], errors="coerce") if DOB else pd.Series(pd.NaT, index=df.index)
df["dob"] = dob.dt.strftime("%Y-%m-%d").fillna("")
df["dob_ok"] = dob.notna() & (dob.dt.year >= 1900) & (dob <= df["call_dt"])
ln_per_dob = df[df["dob_ok"] & df["name_ok"]].groupby("dob")["ln"].nunique()
placeholder_dobs = set(ln_per_dob[ln_per_dob > MAX_LAST_NAMES_PER_DOB].index) | {"1900-01-01","1901-01-01"}
df.loc[df["dob"].isin(placeholder_dobs), "dob_ok"] = False

def norm_phone(x):
    if pd.isna(x): return ""
    s = str(int(x)) if isinstance(x, (int, float, np.integer, np.floating)) else str(x)
    d = re.sub(r"\D", "", s)
    if len(d) == 11 and d[0] == "1": d = d[1:]
    ok = len(d) == 10 and d[0] not in "01" and len(set(d)) > 2 and d != "1234567890"
    return d if ok else ""

df["phone_n"] = df[PHONE].apply(norm_phone) if PHONE else ""
df["phone_ok"] = df["phone_n"] != ""
person_guess = df["fn_canon"] + "|" + df["ln"] + "|" + df["dob"]
ppl_per_phone = df[df["phone_ok"] & df["name_ok"]].assign(pg=person_guess).groupby("phone_n")["pg"].nunique()
shared_phones = set(ppl_per_phone[ppl_per_phone > MAX_PEOPLE_PER_PHONE].index)
df.loc[df["phone_n"].isin(shared_phones), "phone_ok"] = False

df["identifiable"] = df["name_ok"] & (df["dob_ok"] | df["phone_ok"])
print(f"placeholder dates of birth excluded: {len(placeholder_dobs):,}")
print(f"shared phones excluded (more than {MAX_PEOPLE_PER_PHONE} different people): {len(shared_phones):,}")

top_dob = (df[df["dob"] != ""].groupby("dob").agg(calls=("dob","size"), distinct_last_names=("ln","nunique"))
           .sort_values("distinct_last_names", ascending=False).head(10))
top_dob["excluded_as_placeholder"] = top_dob.index.isin(placeholder_dobs)
display(top_dob)

quality = pd.DataFrame({
    "measure":["calls after incident de-duplication","valid first and last name","valid date of birth","valid phone (not shared)",
               "identifiable (valid name plus date of birth or phone)","not identifiable (excluded from repeat measures)"],
    "calls":[len(df), int(df["name_ok"].sum()), int(df["dob_ok"].sum()), int(df["phone_ok"].sum()),
             int(df["identifiable"].sum()), int((~df["identifiable"]).sum())]})
quality["percent_of_calls"] = (quality["calls"] / len(df) * 100).round(1)
keep(quality, "identity_quality")
quality

## 6. Link calls to persons

Rules are applied in order. Each rule links calls that the earlier rules did not already join.

| Rule | Link condition |
|---|---|
| 1 | Same first name, last name, and date of birth |
| 2 | Same last name and date of birth; first name matches by nickname, by spelling similarity with the same first letter, or one is the start of the other (Chris, Christopher) |
| 3 | Same first name and date of birth; last name matches by spelling similarity or one contains the other (hyphenated names) |
| 4 | Same date of birth and phone; first name matches under the rule 2 test, last name may differ (covers a last-name change) |
| 5 | Same first name, last name, and phone, where at most one valid date of birth exists among those calls |

Spelling similarity is the Jaro-Winkler score, a standard name-comparison measure from 0 to 1; the threshold is 0.88 (for example, smith and smyth score 0.89). Names are never linked on name alone. Calls with the same name but different valid dates of birth stay separate (for example, a parent and child with the same name).

In [ ]:
ident = df[df["identifiable"]].reset_index(drop=True)
n = len(ident)
parent = np.arange(n)
def find(x):
    root = x
    while parent[root] != root: root = parent[root]
    while parent[x] != root: parent[x], x = root, parent[x]
    return root

T0 = "0 personal identification number"
T1 = "1 exact name and date of birth"
T2 = "2 similar first name, same last name and date of birth"
T3 = "3 similar last name, same first name and date of birth"
T4 = "4 same first name, date of birth, and phone"
T5 = "5 same name and phone, one date of birth or none"
merges = {t: 0 for t in [T0,T1,T2,T3,T4,T5]}
qa_links = []
def union(a, b, tier):
    ra, rb = find(a), find(b)
    if ra == rb: return
    parent[rb] = ra; merges[tier] += 1
    if tier != T1: qa_links.append((a, b, tier))

def sim(a, b):
    if a == b: return 1.0
    la, lb = len(a), len(b)
    if not la or not lb: return 0.0
    r = max(max(la, lb) // 2 - 1, 0)
    ma, mb, m = [False]*la, [False]*lb, 0
    for i, ch in enumerate(a):
        for j in range(max(0, i - r), min(lb, i + r + 1)):
            if not mb[j] and b[j] == ch:
                ma[i] = mb[j] = True; m += 1; break
    if not m: return 0.0
    t, k = 0, 0
    for i in range(la):
        if ma[i]:
            while not mb[k]: k += 1
            if a[i] != b[k]: t += 1
            k += 1
    jaro = (m / la + m / lb + (m - t / 2) / m) / 3
    p = 0
    for x, y in zip(a[:4], b[:4]):
        if x != y: break
        p += 1
    return jaro + p * 0.1 * (1 - jaro)
def first_match(a, b):
    if NICKNAMES.get(a, a) == NICKNAMES.get(b, b): return True
    if a[:1] != b[:1]: return False
    short, long_ = sorted([a, b], key=len)
    return sim(a, b) >= NAME_MIN_SIMILARITY or (len(short) >= 3 and long_.startswith(short))
def last_match(a, b):
    short, long_ = sorted([a, b], key=len)
    return sim(a, b) >= NAME_MIN_SIMILARITY or (len(short) >= 4 and short in long_)

if USE_PERSONAL_ID and PID:
    pid_ok = ~is_blank(ident[PID])
    for g in ident[pid_ok].groupby(ident.loc[pid_ok, PID].astype(str).str.strip()).groups.values():
        g = list(g)
        for j in g[1:]: union(g[0], j, T0)

has_dob = ident["dob_ok"]
for g in ident[has_dob].groupby(["fn","ln","dob"]).groups.values():
    g = list(g)
    for j in g[1:]: union(g[0], j, T1)

def fuzzy_pass(frame, block_cols, compare_col, matcher, tier):
    multi = frame.groupby(block_cols)[compare_col].transform("nunique") > 1
    for _, g in frame[multi].groupby(block_cols):
        reps = g.drop_duplicates(compare_col)
        idx, vals = list(reps.index), list(reps[compare_col])
        for i in range(len(idx)):
            for j in range(i + 1, len(idx)):
                if matcher(vals[i], vals[j]): union(idx[i], idx[j], tier)

fuzzy_pass(ident[has_dob], ["ln","dob"], "fn", first_match, T2)
fuzzy_pass(ident[has_dob], ["fn_canon","dob"], "ln", last_match, T3)
ident["full_name"] = ident["fn"] + "|" + ident["ln"]
fuzzy_pass(ident[has_dob & ident["phone_ok"]], ["dob","phone_n"], "full_name",
           lambda a, b: first_match(a.split("|")[0], b.split("|")[0]), T4)

np_frame = ident[ident["phone_ok"]]
np_frame = np_frame[np_frame.groupby(["fn_canon","ln","phone_n"])["fn"].transform("size") > 1]
for _, g in np_frame.groupby(["fn_canon","ln","phone_n"]):
    if g.loc[g["dob_ok"], "dob"].nunique() <= 1:
        idx = list(g.index)
        for j in idx[1:]: union(idx[0], j, T5)

ident["cluster"] = [find(i) for i in range(n)]
first_seen = ident.groupby("cluster")["call_dt"].min().sort_values()
key_map = {c: f"P{i+1:06d}" for i, c in enumerate(first_seen.index)}
ident["person_key"] = ident["cluster"].map(key_map)

tiers = pd.DataFrame({"rule": list(merges.keys()), "links_added": list(merges.values())})
tiers = tiers[(tiers["rule"] != T0) | USE_PERSONAL_ID]
keep(tiers, "match_rules")
n_persons = ident["person_key"].nunique()
print(f"identifiable calls: {n:,}; distinct persons: {n_persons:,}; average calls per person: {n/n_persons:.2f}")
tiers

## 7. Link quality checks

A sample of links from the fuzzy rules (2 to 5), showing which fields agree and how similar the names are. Record numbers allow each pair to be looked up in the source data. Names are not displayed. The cluster check lists the largest persons by call count; a person with more than one distinct valid date of birth would indicate an incorrect link.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
qa = pd.DataFrame(qa_links, columns=["a","b","rule"])
if len(qa):
    per_rule = max(1, QA_SAMPLE_N // max(1, qa["rule"].nunique()))
    qa = pd.concat([g.sample(min(len(g), per_rule), random_state=RANDOM_SEED) for _, g in qa.groupby("rule")])
    A, B = ident.loc[qa["a"]].reset_index(drop=True), ident.loc[qa["b"]].reset_index(drop=True)
    qa_out = pd.DataFrame({
        "rule": qa["rule"].values,
        "record_a": A["record_id"], "record_b": B["record_id"],
        "first_name_similarity": [round(sim(x, y), 2) for x, y in zip(A["fn"], B["fn"])],
        "last_name_similarity": [round(sim(x, y), 2) for x, y in zip(A["ln"], B["ln"])],
        "same_date_of_birth": (A["dob"] == B["dob"]) & A["dob_ok"] & B["dob_ok"],
        "same_phone": (A["phone_n"] == B["phone_n"]) & A["phone_ok"] & B["phone_ok"],
        "days_apart": ((B["call_dt"] - A["call_dt"]).abs().dt.total_seconds() / 86400).round(1)})
else:
    qa_out = pd.DataFrame(columns=["rule","record_a","record_b"])
keep(qa_out, "match_qa_sample")
display(qa_out.head(30))

ident["dob_valid"] = ident["dob"].where(ident["dob_ok"])
ident["phone_valid"] = ident["phone_n"].where(ident["phone_ok"])
clus = ident.groupby("person_key").agg(calls=("record_id","size"), distinct_first_names=("fn","nunique"),
        distinct_last_names=("ln","nunique"), distinct_valid_dob=("dob_valid","nunique"),
        distinct_phones=("phone_valid","nunique"))
multi_dob = int((clus["distinct_valid_dob"] > 1).sum())
print(f"persons with more than one distinct valid date of birth: {multi_dob:,}")
largest = clus.sort_values("calls", ascending=False).head(15).reset_index()
keep(largest, "largest_persons")
largest

## 8. Behavioral health keyword flag

Each call is flagged with the same whole-word keyword list and negation handling used in the behavioral health screen, so repeat rates can be compared for calls with and without behavioral health language. The flag is keyword-based and is not validated in this notebook.

In [ ]:
kw_path = os.path.join(PROMPT_DIR, "bh_keywords.txt")
NEGATIONS = ["denies","denied","no ","without","negative for","ruled out","not ","non-"]
if NOTES and os.path.exists(kw_path):
    with open(kw_path) as f:
        kws = [k.strip().lower() for k in f.read().splitlines() if k.strip() and not k.strip().startswith("#")]
    pats = [re.compile("(?<![a-z0-9])" + re.escape(k) + "(?:s|es)?(?![a-z0-9])") for k in kws]
    def kw_hit(text):
        t = str(text).lower()
        for p in pats:
            for m in p.finditer(t):
                if not any(neg in t[max(0, m.start()-15):m.start()] for neg in NEGATIONS): return True
        return False
    ident["bh_flag"] = ident[NOTES].fillna("").apply(kw_hit)
    print(f"keywords loaded: {len(kws)}; identifiable calls with behavioral health keyword matches: {int(ident['bh_flag'].sum()):,} ({ident['bh_flag'].mean()*100:.1f}%)")
else:
    ident["bh_flag"] = False
    print("keyword file or notes column not found; behavioral health flag not applied")

## 9. Return rates within 30, 60, and 90 days

In [ ]:
d = ident.sort_values(["person_key","call_dt"]).reset_index(drop=True)
g = d.groupby("person_key")["call_dt"]
d["next_gap_days"] = (g.shift(-1) - d["call_dt"]).dt.total_seconds() / 86400
data_end = d["call_dt"].max()
for w in WINDOWS:
    d[f"eligible_{w}"] = d["call_dt"] <= data_end - pd.Timedelta(days=w)
    d[f"return_{w}"] = d["next_gap_days"].le(w)

rows = []
for w in WINDOWS:
    e = d[d[f"eligible_{w}"]]
    persons_e = e["person_key"].nunique()
    persons_r = e.loc[e[f"return_{w}"], "person_key"].nunique()
    rows.append({"window_days": w, "eligible_index_calls": len(e), "calls_with_return": int(e[f"return_{w}"].sum()),
                 "return_rate_percent": round(e[f"return_{w}"].mean()*100, 1),
                 "persons": persons_e, "persons_with_a_return": persons_r,
                 "persons_with_a_return_percent": round(persons_r / persons_e * 100, 1) if persons_e else 0})
rates = pd.DataFrame(rows)
keep(rates, "return_rates")

fig, ax = plt.subplots(figsize=(8,4))
ax.bar([f"{w} days" for w in WINDOWS], rates["return_rate_percent"], color=[TEAL, NAVY, GREY])
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title("Share of calls followed by another call from the same person")
for i, v in enumerate(rates["return_rate_percent"]): ax.annotate(f"{v:.1f}%", (i, v), xytext=(0,4), textcoords="offset points", ha="center")
plt.tight_layout(); plt.show()
rates

In [ ]:
bins = [-0.001, 1, 7, 30, 60, 90, 180, np.inf]
labels = ["1 day or less","2-7 days","8-30 days","31-60 days","61-90 days","91-180 days","over 180 days"]
gaps = pd.cut(d["next_gap_days"].dropna(), bins=bins, labels=labels).value_counts().reindex(labels)
gap_df = gaps.rename("calls").to_frame()
gap_df["percent_of_calls_with_a_next_call"] = (gap_df["calls"] / gap_df["calls"].sum() * 100).round(1)
keep(gap_df.reset_index().rename(columns={"index":"days_to_next_call","next_gap_days":"days_to_next_call"}), "days_to_next_call")

fig, ax = plt.subplots(figsize=(10,4))
ax.bar(range(len(gap_df)), gap_df["calls"], color=TEAL)
ax.set_xticks(range(len(gap_df))); ax.set_xticklabels(labels, fontsize=9)
ax.set_title("Days from a call to the same person's next call")
for i, v in enumerate(gap_df["calls"]): ax.annotate(f"{v:,}", (i, v), xytext=(0,4), textcoords="offset points", ha="center", fontsize=9)
plt.tight_layout(); plt.show()
gap_df

## 10. Calls per person and frequent callers

Frequent callers are counted by the highest number of calls a person made inside any rolling 30, 60, or 90 day period.

In [ ]:
per = d.groupby("person_key").agg(calls=("call_dt","size"), first_call=("call_dt","min"), last_call=("call_dt","max"),
                                  behavioral_health_keyword_calls=("bh_flag","sum"))
def max_in_window(t, w):
    s = np.sort(t.values.astype("datetime64[s]").astype(np.int64))
    return int((np.searchsorted(s, s + w*86400, side="right") - np.arange(len(s))).max())
repeaters = d[d["person_key"].isin(per.index[per["calls"] > 1])]
for w in WINDOWS:
    m = repeaters.groupby("person_key")["call_dt"].apply(lambda t: max_in_window(t, w))
    per[f"max_calls_in_{w}_days"] = m.reindex(per.index).fillna(1).astype(int)

buckets = pd.cut(per["calls"], bins=[0,1,2,4,9,np.inf], labels=["1","2","3-4","5-9","10 or more"])
cpp = per.groupby(buckets).agg(persons=("calls","size"), calls=("calls","sum"))
cpp["percent_of_persons"] = (cpp["persons"] / cpp["persons"].sum() * 100).round(1)
cpp["percent_of_calls"] = (cpp["calls"] / cpp["calls"].sum() * 100).round(1)
cpp.index.name = "calls_per_person"
keep(cpp.reset_index(), "calls_per_person")

freq_rows = []
for w in WINDOWS:
    for k in FREQUENT_THRESHOLDS:
        sel = per[per[f"max_calls_in_{w}_days"] >= k]
        freq_rows.append({"window_days": w, "minimum_calls_in_window": k, "persons": len(sel),
                          "percent_of_persons": round(len(sel) / len(per) * 100, 2),
                          "total_calls_from_these_persons": int(sel["calls"].sum()),
                          "percent_of_calls": round(sel["calls"].sum() / per["calls"].sum() * 100, 1)})
freq = pd.DataFrame(freq_rows)
keep(freq, "frequent_callers")

fig, ax = plt.subplots(figsize=(9,4))
x = np.arange(len(cpp)); wd = 0.38
ax.bar(x - wd/2, cpp["percent_of_persons"], wd, color=TEAL, label="% of persons")
ax.bar(x + wd/2, cpp["percent_of_calls"], wd, color=NAVY, label="% of calls")
ax.set_xticks(x); ax.set_xticklabels(cpp.index.astype(str)); ax.set_xlabel("calls per person")
ax.yaxis.set_major_formatter(mtick.PercentFormatter()); ax.legend()
ax.set_title("Calls per person - share of persons and share of calls")
plt.tight_layout(); plt.show()
display(cpp); display(freq)

In [ ]:
top = per.sort_values("calls", ascending=False).head(25).copy()
top_rows = d[d["person_key"].isin(top.index)]
mode_of = lambda s: s.astype(str).mode().iloc[0] if len(s.dropna()) else ""
top_market = top_rows.groupby("person_key")[MARKET].agg(mode_of) if MARKET else None
top_dispo = top_rows.groupby("person_key")[DISPO].agg(mode_of) if DISPO else None
top["span_days"] = (top["last_call"] - top["first_call"]).dt.days
if MARKET: top["most_common_market"] = top_market.reindex(top.index)
if DISPO: top["most_common_disposition"] = top_dispo.reindex(top.index)
top["first_call"] = top["first_call"].dt.strftime("%Y-%m-%d"); top["last_call"] = top["last_call"].dt.strftime("%Y-%m-%d")
top = top.reset_index()
keep(top, "top_frequent_callers")
top

## 11. Return rates by market, disposition, caller type, and behavioral health flag

Each rate uses its own eligible index calls, so market and disposition rates are normalized to their own call volume.

In [ ]:
def return_by(col, label, top_n=None):
    key = d[col].fillna("(blank)").astype(str) if col in d.columns else pd.Series("(blank)", index=d.index)
    out = None
    for w in WINDOWS:
        e = d[f"eligible_{w}"]
        t = d[e].groupby(key[e]).agg(**{f"eligible_calls_{w}_days": (f"return_{w}","size"),
                                        f"return_rate_{w}_days_percent": (f"return_{w}","mean")})
        t[f"return_rate_{w}_days_percent"] = (t[f"return_rate_{w}_days_percent"] * 100).round(1)
        out = t if out is None else out.join(t, how="outer")
    out = out.sort_values(f"eligible_calls_{WINDOWS[0]}_days", ascending=False)
    if top_n: out = out.head(top_n)
    out.index.name = label
    return out.reset_index()

def rate_chart(t, label, title):
    o = t.sort_values(f"return_rate_{WINDOWS[0]}_days_percent")
    fig, ax = plt.subplots(figsize=(10, max(3, 0.35*len(o) + 1)))
    ax.barh(range(len(o)), o[f"return_rate_{WINDOWS[0]}_days_percent"], color=TEAL)
    ax.set_yticks(range(len(o))); ax.set_yticklabels([str(v)[:45] for v in o[label]], fontsize=9)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter()); ax.set_title(title)
    for i, v in enumerate(o[f"return_rate_{WINDOWS[0]}_days_percent"]):
        ax.annotate(f"{v:.1f}%", (v, i), xytext=(4,0), textcoords="offset points", va="center", fontsize=8)
    plt.tight_layout(); plt.show()

by_market = keep(return_by(MARKET, "market"), "by_market") if MARKET else None
if by_market is not None: rate_chart(by_market, "market", f"{WINDOWS[0]}-day return rate by market"); display(by_market)

In [ ]:
by_dispo = keep(return_by(DISPO, "disposition", top_n=15), "by_disposition") if DISPO else None
if by_dispo is not None: rate_chart(by_dispo, "disposition", f"{WINDOWS[0]}-day return rate by disposition of the index call (15 most common)"); display(by_dispo)

In [ ]:
by_caller = keep(return_by(CALLER_TYPE, "caller_type"), "by_caller_type") if CALLER_TYPE else None
if by_caller is not None: display(by_caller)
d["behavioral_health_keyword_match"] = np.where(d["bh_flag"], "keyword match", "no keyword match")
by_bh = keep(return_by("behavioral_health_keyword_match", "behavioral_health_keyword_match"), "by_behavioral_health")
display(by_bh)

## 12. Monthly trend

The 30-day return rate by month of the index call. Months without a full 30-day follow-up window before the end of the data are excluded.

In [ ]:
w = WINDOWS[0]
d["month"] = d["call_dt"].dt.to_period("M")
full = d["month"].dt.end_time <= data_end - pd.Timedelta(days=w)
trend = d[full].groupby("month").agg(index_calls=(f"return_{w}","size"), calls_with_return=(f"return_{w}","sum"))
trend["return_rate_percent"] = (trend["calls_with_return"] / trend["index_calls"] * 100).round(1)
trend.index = trend.index.astype(str)
keep(trend.reset_index(), "return_trend")
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(range(len(trend)), trend["return_rate_percent"], marker="o", lw=2, color=TEAL)
ax.set_xticks(range(len(trend))); ax.set_xticklabels(trend.index, rotation=60, fontsize=8)
ax.set_ylim(0, max(10, trend["return_rate_percent"].max() * 1.2) if len(trend) else 10)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title(f"{w}-day return rate by month of index call"); ax.set_ylabel("% of calls")
plt.tight_layout(); plt.show()
trend

## 13. Write the Excel output

In [ ]:
def sanitize(x):
    o = x.copy(); o.columns = [str(c) for c in o.columns]
    for c in o.columns:
        if o[c].dtype == object or str(o[c].dtype).startswith("category"):
            o[c] = o[c].apply(lambda v: "" if v is None or (isinstance(v, float) and pd.isna(v)) else str(v))
    return o
TABS = [("field_mapping","Field Mapping"),("identity_profile","Identity Field Profile"),("identity_quality","Identity Quality"),
        ("match_rules","Match Rules"),("match_qa_sample","Link Review Sample"),("largest_persons","Largest Persons"),
        ("return_rates","Return Rates"),("days_to_next_call","Days to Next Call"),("calls_per_person","Calls per Person"),
        ("frequent_callers","Frequent Callers"),("top_frequent_callers","Top Frequent Callers"),("by_market","By Market"),
        ("by_disposition","By Disposition"),("by_caller_type","By Caller Type"),("by_behavioral_health","By Behavioral Health"),
        ("return_trend","Return Trend")]
xlsx = os.path.join(OUT_DIR, f"Nurse_Navigation_Repeat_Callers_{RUN_ID}.xlsx")
try: import xlsxwriter; eng = "xlsxwriter"
except ImportError: eng = "openpyxl"
start = [f"Run {RUN_ID}", f"Source: {SOURCE_FILE}",
         f"Call dates: {df['call_dt'].min():%Y-%m-%d} to {df['call_dt'].max():%Y-%m-%d}",
         f"Calls after incident de-duplication: {len(df):,}; identifiable: {len(ident):,}; distinct persons: {n_persons:,}",
         "Repeat (return): a separate 911 incident from the same person within 30, 60, or 90 days of an index call.",
         "Return rates count only index calls with a full follow-up window before the end of the data.",
         "Persons are resolved from patient first name, last name, date of birth, and phone. See Match Rules.",
         "Behavioral health flag: whole-word keyword match from the behavioral health screen; not validated here.",
         "No names, dates of birth, or phone numbers are included. Persons are labeled with a generated person key."]
with pd.ExcelWriter(xlsx, engine=eng) as wtr:
    pd.DataFrame({"Nurse Navigation - Repeat Callers": start}).to_excel(wtr, sheet_name="Start Here", index=False)
    for stem, tab in TABS:
        x = RESULTS.get(stem)
        if x is not None and len(x): sanitize(x).to_excel(wtr, sheet_name=tab[:31], index=False); print("added", tab)
for old in glob.glob(os.path.join(OUT_DIR, "Nurse_Navigation_Repeat_Callers_*.xlsx")):
    if os.path.abspath(old) != os.path.abspath(xlsx):
        try: os.remove(old)
        except Exception: pass
print("output:", os.path.basename(xlsx))